In [3]:
# Stop warnings
import warnings
warnings.filterwarnings("ignore")

# Imports
import os
import sys
import time
import json
import copy
import cortex
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# Plotly imports
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Personal imports
sys.path.append("/Users/sinakling/disks/meso_H/projects/pRF_analysis/analysis_code/utils")
from surface_utils import load_surface
from settings_utils import load_settings
from maths_utils import weighted_regression
from plot_utils import plotly_template, active_vertex_roi_mmp_plot, prf_contralaterality_plot
from pycortex_utils import draw_cortex, set_pycortex_config_file, load_surface_pycortex, create_colormap, get_rois

In [4]:
# Directories
main_dir = '/Users/sinakling/disks/meso_shared'
project_dir = 'RetinoMaps'
rois_method_format = 'rois-group-mmp'
rsq2use = 'prf_rsq'

In [6]:
# Load settings
base_dir = "/Users/sinakling/disks/meso_H/projects/pRF_analysis"
settings_path = os.path.join(base_dir, project_dir, "settings.yml")
prf_settings_path = os.path.join(base_dir, project_dir, "pmf-analysis.yml")
figure_settings_path = os.path.join(base_dir, project_dir, "figure-settings.yml")
settings = load_settings([settings_path, prf_settings_path, figure_settings_path])
analysis_info = settings[0]

In [7]:
if rois_method_format == 'rois-drawn':
    analysis_info['rois'] = analysis_info[rois_method_format]
elif rois_method_format == 'rois-group-mmp':
    analysis_info['rois'] = list(analysis_info[rois_method_format].keys())

In [8]:
# Settings
subject = 'group'
format_ = '170k'

analysis_info['rois_groups_plot'] = [['iIPS', 'sIPS'],['iPCS', 'sPCS', 'mPCS']]
analysis_info['rois'] = ['iIPS', 'sIPS', 'iPCS', 'sPCS', 'mPCS']

In [9]:
# General settings
TRs = analysis_info['TRs']
subjects = analysis_info['subjects']
rois_groups = analysis_info['rois-group-mmp']

plot_groups = analysis_info['rois_groups_plot']
fig_width = 1440
ecc_threshold = analysis_info['ecc_th']
size_threshold = analysis_info['size_th']
rsqr_threshold = analysis_info['rsqr_th']
amplitude_threshold = analysis_info['prf_amp_th']
stats_threshold = analysis_info['stats_th']
n_threshold = analysis_info['n_th']

In [17]:
df_stats = pd.DataFrame()
# for subject in subjects 
for subject in subjects :
    df_ecc_size = pd.read_table(f"/Users/sinakling/disks/meso_shared/RetinoMaps/derivatives/pp_data/{subject}/170k/pmf/tsv/{subject}_task-SacLoc_fmriprep_dct_z-score_concat_rois-group-mmp_pmf-css-odm_ecc-size.tsv")
    # for roi in rois: 
    for roi in ['iIPS', 'sIPS', 'iPCS', 'sPCS', 'mPCS']: 
        df_stats_roi = pd.DataFrame()
        # Get data
        df_ecc_size_roi = df_ecc_size.loc[(df_ecc_size.roi == roi)]
        
        ecc_median = np.array(df_ecc_size_roi.prf_ecc_bins)
        size_median = np.array(df_ecc_size_roi.prf_size_bins_median)
        r2_median = np.array(df_ecc_size_roi.prf_rsq_bins_median)
        
        # Creat model 
        slope, intercept = weighted_regression(ecc_median, size_median, r2_median, model='linear')
        
        # Load data
        df_pRF = pd.read_table(f"/Users/sinakling/disks/meso_shared/RetinoMaps/derivatives/pp_data/{subject}/170k/pmf/tsv/{subject}_task-SacLoc_fmriprep_dct_z-score_concat_rois-group-mmp_pmf-css-odm_deriv.tsv")
        
        # Threshold data (replace by nan)
        if stats_threshold == 0.05: stats_col = 'corr_pvalue_5pt'
        elif stats_threshold == 0.01: stats_col = 'corr_pvalue_1pt'
        df_pRF.loc[(df_pRF.amplitude < amplitude_threshold[0]) |
                 (df_pRF.prf_ecc < ecc_threshold[0]) | (df_pRF.prf_ecc > ecc_threshold[1]) |
                 (df_pRF.prf_size < size_threshold[0]) | (df_pRF.prf_size > size_threshold[1]) |
                 (df_pRF.prf_n < n_threshold[0]) | (df_pRF.prf_n > n_threshold[1]) | 
                 (df_pRF[rsq2use] < rsqr_threshold) |
                 (df_pRF[stats_col] > stats_threshold)] = np.nan
        df_pRF = df_pRF.dropna()
        
        
        df_pRF_roi = df_pRF.loc[(df_pRF.roi == roi)]
        
        pred_size_roi = df_pRF_roi.prf_ecc * slope + intercept 
        
        # mask for nan
        mask = ~df_pRF_roi.prf_size.isna() & ~pred_size_roi.isna()

        # linear regression between model and data 
        results = stats.linregress(x=pred_size_roi[mask], y=df_pRF_roi.prf_size[mask], alternative='greater')
        
        df_stats_roi['subject'] = [subject]
        df_stats_roi['roi'] = [roi]
        df_stats_roi['rvalue'] = [results.rvalue]
        df_stats_roi['pvalue'] = [results.pvalue]
        
        df_stats = pd.concat([df_stats, df_stats_roi])
        

In [18]:
fig = go.Figure()
fig.add_trace(go.Table(header=dict(values=list(df_stats.columns), 
                                   fill_color='paleturquoise', 
                                   align='left'), 
                       cells=dict(values=[df_stats[col].round(3) for col in df_stats.columns], 
                                  fill_color='lavender', 
                                  align='left')
                      ))

fig.update_layout(template='simple_white',
                  height=2500, 
                  width=1200)

In [19]:
print('ecc / size relation signifante subjects')
for roi in ['iIPS', 'sIPS', 'iPCS', 'sPCS', 'mPCS'] : 
    df_stats_roi_significant = df_stats.loc[(df_stats['roi']==roi)&(df_stats['pvalue'] < 0.01)]
    
    print('{}'. format(roi))
    print('all p-value inferior to {}'.format(df_stats_roi_significant['pvalue'].max()))
    print('r interval : {} {}'.format(df_stats_roi_significant['rvalue'].min().round(2), df_stats_roi_significant['rvalue'].max().round(2)))

ecc / size relation signifante subjects
iIPS
all p-value inferior to 1.0585925552770119e-33
r interval : 0.44 0.77
sIPS
all p-value inferior to 2.9478501044867042e-53
r interval : 0.31 0.8
iPCS
all p-value inferior to 5.241013079415827e-25
r interval : 0.26 0.78
sPCS
all p-value inferior to 7.71555732555841e-30
r interval : 0.31 0.72
mPCS
all p-value inferior to 0.009029252547753195
r interval : 0.1 0.86


In [20]:
print('ecc / size relation non signifante subjects')
for roi in ['iIPS', 'sIPS', 'iPCS', 'sPCS', 'mPCS']:
    df_stats_roi_significant = df_stats.loc[(df_stats['roi'] == roi) & (df_stats['pvalue'] > 0.01)]
    
    if df_stats_roi_significant.empty:
        print('{}: no subject non significant'.format(roi))
        continue  # Passer à l'itération suivante si df_stats_roi_significant est vide
    
    print('{}'.format(roi))
    print('{} subject(s) is / are not significative : ({})'.format(df_stats_roi_significant['subject'].shape[0] , df_stats_roi_significant['subject'].values))
    print('all p-value superior to {}'.format(df_stats_roi_significant['pvalue'].min()))
    print('r interval : {} {}'.format(df_stats_roi_significant['rvalue'].min().round(2), df_stats_roi_significant['rvalue'].max().round(2)))


ecc / size relation non signifante subjects
iIPS
1 subject(s) is / are not significative : (['sub-14'])
all p-value superior to 1.0
r interval : -0.48 -0.48
sIPS: no subject non significant
iPCS: no subject non significant
sPCS: no subject non significant
mPCS: no subject non significant
